# ⚡ Day 145 — Advanced Caching + Multi-Page Streamlit Apps
## Month 8 | Deployment & App Building | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 145 (Month 8, Day 5 — Week 1 Closer) |
| **Topic** | `@st.cache_data` vs `@st.cache_resource` · TTL · Cache clearing · Multi-page app architecture |
| **Dataset** | FreelanceHub India (500 rows, seed=141) |
| **Deliverable** | `day145_app/` folder — 3-page professional Streamlit app |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

> **Why this matters for freelancing:**
> Single-page Streamlit scripts feel like demos. Multi-page apps feel like products.
> Clients on Upwork consistently pay 2–3× more for "a proper app with navigation" vs "a Python script with charts."
> Caching is what makes those apps fast enough to demo live on a client call — without it, every filter click reloads 500 rows from disk.

---

## Month 8 Week 1 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | Pending |
| 144 | ML Model Deployment | Pending |
| **145** | **Advanced Caching + Multi-Page Apps** | **Today** |


---
## 📂 Section 1 — Raw Data
> ⚠️ **NEVER MODIFY THIS SECTION.** Copy this data generator into your app files. Never edit seed or column names.


In [1]:
# ── Section 1: Raw Data Generator ──────────────────────────────────────────
# Comment: Reproducible synthetic dataset — FreelanceHub India, seed=141, 500 rows
# Method:  numpy random with fixed seed; intentional nulls for realism

import pandas as pd
import numpy as np

def generate_data(seed=141, n=500):
    np.random.seed(seed)
    categories = ['ML/AI', 'Web Dev', 'Data Analytics',
                  'Graphic Design', 'Content Writing', 'SEO/Digital Marketing']
    platforms  = ['Upwork', 'Freelancer', 'Fiverr', 'Toptal', 'Direct Client']
    cities     = ['Mumbai', 'Bangalore', 'Delhi', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
    levels     = ['Entry', 'Mid', 'Senior']
    outcomes   = ['Completed', 'Cancelled', 'In Progress']
    months     = ['Jan','Feb','Mar','Apr','May','Jun',
                  'Jul','Aug','Sep','Oct','Nov','Dec']

    df = pd.DataFrame({
        'project_id':    [f'PJ{1000+i}' for i in range(n)],
        'category':      np.random.choice(categories, n),
        'platform':      np.random.choice(platforms, n, p=[0.30,0.25,0.25,0.10,0.10]),
        'city':          np.random.choice(cities, n),
        'level':         np.random.choice(levels, n, p=[0.40,0.40,0.20]),
        'budget_inr':    np.random.randint(5000, 120001, n),
        'duration_days': np.random.randint(7, 91, n).astype(float),
        'client_rating': np.round(np.random.uniform(2.5, 5.0, n), 1),
        'bids_received': np.random.randint(2, 51, n).astype(float),
        'outcome':       np.random.choice(outcomes, n, p=[0.65, 0.20, 0.15]),
        'month_posted':  np.random.choice(months, n),
    })
    # Intentional nulls
    null_idx = np.random.choice(n, 25, replace=False)
    df.loc[null_idx[:10], 'client_rating'] = np.nan
    df.loc[null_idx[10:20], 'bids_received'] = np.nan
    df.loc[null_idx[20:], 'duration_days'] = np.nan
    return df

# Verify — run this cell to confirm raw data
df_raw = generate_data()
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"Null counts:\n{df_raw.isnull().sum()[df_raw.isnull().sum()>0]}")
print("\nFirst 3 rows:")
print(df_raw.head(3).to_string())


Shape: (500, 11)
Columns: ['project_id', 'category', 'platform', 'city', 'level', 'budget_inr', 'duration_days', 'client_rating', 'bids_received', 'outcome', 'month_posted']
Null counts:
duration_days     5
client_rating    10
bids_received    10
dtype: int64

First 3 rows:
  project_id        category       platform     city  level  budget_inr  duration_days  client_rating  bids_received    outcome month_posted
0     PJ1000         Web Dev         Fiverr     Pune  Entry      112435           17.0            2.8           29.0  Completed          Aug
1     PJ1001         Web Dev         Upwork  Chennai    Mid       95517           13.0            4.2           15.0  Completed          May
2     PJ1002  Graphic Design  Direct Client     Pune    Mid       71076           62.0            4.8           19.0  Cancelled          Jan


---
## 📚 Section 2 — Concept Notes

### The Streamlit Re-Run Problem

Every time a user clicks a button, moves a slider, or selects a dropdown, Streamlit **reruns the entire script from top to bottom.** Without caching, this means:
- 500 rows loaded from disk → every click
- A trained Random Forest reloaded → every click  
- A database query fired → every click

**Caching is the fix.** It stores function outputs in memory and returns them instantly on subsequent calls.

---

### `@st.cache_data` — For Data Objects

```python
@st.cache_data
def load_data():
    return pd.read_csv("data.csv")   # runs ONCE; cached result returned on reruns
```

**Use for:** DataFrames, plots (as bytes), API responses, any pure data transformation.  
**Key behaviour:** Each call with different arguments gets its own cache entry.  
**Mutation warning:** If you mutate the returned DataFrame, Streamlit will warn you — always work on a copy.

```python
df = load_data()
df_filtered = df[df['platform'] == 'Upwork']   # ✅ copy — safe
df['new_col'] = 1                               # ⚠️ mutates cache — Streamlit warns
```

---

### `@st.cache_resource` — For Shared Resources

```python
@st.cache_resource
def load_model():
    return joblib.load("model.pkl")   # loads ONCE; all users share this object
```

**Use for:** ML models, database connections, API clients, tokenizers — objects that are expensive to create and safe to share across sessions.  
**Key difference from cache_data:** cache_resource is NOT copied between sessions — all users share one instance. Use when mutation is not a concern (models are read-only).

---

### TTL — Time To Live

```python
@st.cache_data(ttl=3600)      # cache expires after 1 hour
def fetch_live_data():
    return requests.get(API_URL).json()
```

TTL in seconds. After expiry, the next call re-executes the function. Essential for live dashboards where data refreshes on a schedule.

---

### Cache Invalidation

```python
# Clear one function's cache
load_data.clear()

# Clear ALL caches
st.cache_data.clear()
st.cache_resource.clear()
```

Always pair a "🔄 Refresh Data" button with `st.cache_data.clear()` in client dashboards — clients expect fresh data on demand.

---

### Multi-Page App Architecture

**Before Streamlit 1.28:** Use the `pages/` folder pattern.  
**Streamlit 1.28+:** `st.navigation()` API (preferred for new projects).

#### `pages/` folder pattern (backward compatible):
```
my_app/
├── app.py              ← entry point (Home page)
├── pages/
│   ├── 1_📊_Analysis.py
│   └── 2_🤖_Prediction.py
├── utils/
│   └── data_loader.py  ← shared cache functions
└── requirements.txt
```

Streamlit auto-generates sidebar navigation from the `pages/` folder. File names become page labels (underscores → spaces, numbers → sort order).

#### `st.navigation()` API (modern, more control):
```python
# app.py
home = st.Page("pages/home.py", title="Home", icon="🏠")
analysis = st.Page("pages/analysis.py", title="Analysis", icon="📊")
prediction = st.Page("pages/prediction.py", title="Prediction", icon="🤖")

pg = st.navigation([home, analysis, prediction])
pg.run()
```

---

### Shared State Between Pages

Pages cannot share variables directly. Use:
1. `st.session_state` — for user choices (filters, form inputs)
2. Cached functions in `utils.py` — for shared data (both pages call the same `@st.cache_data` function)

```python
# utils/data_loader.py
@st.cache_data
def get_data():
    return generate_data()   # called by any page — returns same cached object

# page1.py
from utils.data_loader import get_data
df = get_data()   # instant — already cached

# page2.py  
from utils.data_loader import get_data
df = get_data()   # same instant — same cache
```

---

### cache_data vs cache_resource — Decision Matrix

| Situation | Decorator |
|---|---|
| Load a CSV / DataFrame | `@st.cache_data` |
| Run a SQL query | `@st.cache_data` |
| Load a trained ML model (.pkl) | `@st.cache_resource` |
| Create a DB connection | `@st.cache_resource` |
| Compute an aggregation | `@st.cache_data` |
| Initialise a tokenizer / embedder | `@st.cache_resource` |
| Data that refreshes hourly | `@st.cache_data(ttl=3600)` |

**Rule of thumb:** Data objects → cache_data. Connection/model objects → cache_resource.


---
## 🛠️ Section 3 — Practice Guide

You will build a **3-page professional Streamlit app** called `day145_app/`.

### Required folder structure:
```
day145_app/
├── app.py                    ← Entry point / Home page
├── pages/
│   ├── 1_📊_Analysis.py      ← Platform & category analysis
│   └── 2_🤖_Prediction.py    ← Budget tier classifier (rule-based)
├── utils/
│   └── data_loader.py        ← Shared cached data functions
└── requirements.txt
```

Run with: `streamlit run day145_app/app.py`

---

### Task 1 — `utils/data_loader.py`: Shared Cached Loader (15 pts)

Build `utils/data_loader.py` with:

**a) `get_raw_data()`** — decorated with `@st.cache_data`, returns the full 500-row FreelanceHub DataFrame. **(5 pts)**

**b) `get_summary_stats(df)`** — decorated with `@st.cache_data`, takes the DataFrame and returns a dict with these 5 keys:
- `total_projects` → int (500)
- `completed_pct` → float (% of Completed outcomes, rounded to 1 decimal)
- `avg_budget` → float (mean budget_inr, rounded to 0 decimal)
- `top_platform` → str (platform with highest avg budget)
- `null_count` → int (total nulls across all columns) **(10 pts)**

Print the dict to verify all 5 values are correct.

**Expected values (from pre-run answer key):**
- total_projects = 500
- completed_pct = 65.0%
- avg_budget = 64,163 ₹
- top_platform = "Fiverr"
- null_count = 25

---

### Task 2 — `app.py`: Home Page with Cache Demo (15 pts)

Build `app.py` with:

**a)** `st.title()` → "FreelanceHub India Dashboard" **(2 pts)**

**b)** Import and call `get_raw_data()` from utils. Display load time using `time.time()` before and after:
```python
import time
t0 = time.time()
df = get_raw_data()
load_time = time.time() - t0
st.write(f"Data loaded in: {load_time:.4f}s")
```
Run the app, refresh the page, and screenshot the second load time (should be < 0.001s — cache hit). **(5 pts)**

**c)** Display the 5 KPI metrics from `get_summary_stats()` using `st.metric()` in a single row (use `st.columns(5)`). **(5 pts)**

**d)** Add a sidebar button "🔄 Refresh Data" that calls `get_raw_data.clear()` and then `st.rerun()`. **(3 pts)**

---

### Task 3 — `pages/1_📊_Analysis.py`: Platform & Category Analysis (20 pts)

Build the Analysis page with:

**a)** Sidebar multiselect for platforms (default = all). Filter the cached df by selected platforms. **(5 pts)**

**b)** Display a bar chart (Plotly or Matplotlib) of **Average Budget by Platform** for the filtered data. Title must include the count of selected platforms. **(7 pts)**

**c)** Display a bar chart of **Project Count by Category** (horizontal bar, sorted descending). **(8 pts)**

> Use `@st.cache_data` to wrap any aggregation function (e.g. `get_platform_summary(df)`) that runs groupby — demonstrate that re-selecting the same platform filter uses cache, not recomputation.

---

### Task 4 — `pages/2_🤖_Prediction.py`: Budget Tier Classifier (15 pts)

Build a rule-based budget tier predictor (no ML model needed — pure business logic):

**a)** Three sidebar number inputs:
- `duration_days` (7–90, default 30)
- `bids_received` (2–50, default 20)  
- `client_rating` (1.0–5.0, step 0.1, default 4.0) **(5 pts)**

**b)** A "Predict Budget Tier" button. On click, compute tier using this rule:
```
If client_rating >= 4.0 AND duration_days >= 30 AND bids_received <= 20:
    tier = "Premium (₹80k–₹1.2L)"
elif client_rating >= 3.5 AND duration_days >= 15:
    tier = "Mid-Range (₹40k–₹80k)"
else:
    tier = "Standard (₹5k–₹40k)"
```
Display tier with `st.success()`, `st.warning()`, or `st.error()` matching Premium/Mid/Standard. **(7 pts)**

**c)** Store the last 5 predictions in `st.session_state['history']` (list of dicts). Display as a table below the prediction button. **(3 pts)**

---

### Task 5 — `requirements.txt` + Interview Question (15 pts)

**a)** Write `requirements.txt` with exact versions for all imports your app uses. Minimum required: streamlit, pandas, numpy, plotly. **(5 pts)**

**b)** In a markdown cell in this notebook, write the answer to:

> "A client says your Streamlit dashboard is slow — every filter takes 3–4 seconds to load. How do you diagnose and fix this?"

**NRA format required.** Must include the specific decorator name, what it does, and how you'd add a cache-clear button. **(10 pts)**

---

### ★ Bonus Task — `st.navigation()` Migration (10 pts)

Rewrite `app.py` to use the modern `st.navigation()` API instead of relying on the `pages/` folder auto-detection:
```python
pg = st.navigation({
    "Home": [st.Page("pages/home.py", title="Home", icon="🏠")],
    "Analytics": [st.Page("pages/1_📊_Analysis.py", ...)],
    "Tools":     [st.Page("pages/2_🤖_Prediction.py", ...)]
})
pg.run()
```
Show a screenshot of the grouped sidebar navigation. **(10★)**


### NRA: Fixing a Slow Streamlit Dashboard (Task 5b)

**Number:**  
Each filter interaction takes 3–4 seconds – confirmed by adding `time.time()` logs that show identical execution time on every click.

**Reason:**  
The root cause is missing `@st.cache_data` decorators. Streamlit re‑runs the entire script top‑to‑bottom after every user action (slider, dropdown, button). Without caching, functions like `pd.read_csv()` and `df.groupby().mean()` execute from scratch each time – the CSV is read from disk, parsed, and aggregated repeatedly.

**Action:**  
1. Decorate all data loading and aggregation functions with `@st.cache_data` in `utils/data_loader.py`.  
2. Add a sidebar refresh button that calls `load_data.clear()` and `st.rerun()` to let the client force a fresh load when needed.  
3. For ML models, use `@st.cache_resource` instead to share one model across all sessions.

After applying this fix, filter interactions drop from 3–4s to <10ms (cache hit).  
Example code:
```python
@st.cache_data
def get_raw_data():
    return pd.read_csv("data.csv")

---
## 📊 Section 4 — Scoring Rubric

| Task | Component | Points | What is checked |
|---|---|---|---|
| **1** | `get_raw_data()` with `@st.cache_data` | 5 | Decorator present; returns correct 500-row df |
| **1** | `get_summary_stats()` — all 5 keys correct | 10 | Values match answer key exactly |
| **2a** | `st.title()` text | 2 | Title matches spec |
| **2b** | Load-time display — 2nd load < 0.001s | 5 | Cache hit demonstrated via screenshot |
| **2c** | 5 KPIs in `st.columns(5)` with `st.metric()` | 5 | All 5 metrics present, values correct |
| **2d** | Sidebar "Refresh" button clears cache + reruns | 3 | `get_raw_data.clear()` + `st.rerun()` both present |
| **3a** | Sidebar multiselect + filter logic | 5 | Correct filter; default = all platforms |
| **3b** | Avg Budget by Platform bar chart | 7 | Plotly or Matplotlib; title includes count; correct data |
| **3c** | Project Count by Category (horizontal, sorted) | 8 | Horizontal orientation; sorted descending |
| **4a** | 3 sidebar inputs with correct ranges | 5 | duration_days, bids_received, client_rating |
| **4b** | Tier classification + correct st.success/warning/error | 7 | Rule logic correct; display function matches tier |
| **4c** | Session state history (last 5 predictions) | 3 | session_state used; displays as table |
| **5a** | `requirements.txt` with correct packages | 5 | streamlit, pandas, numpy, plotly minimum |
| **5b** | NRA cache-debugging answer | 10 | Number (time measurement) + Reason (decorator missing) + Action (specific fix with code reference) |
| **★** | `st.navigation()` migration with screenshot | 10★ | Grouped sidebar shown; pg.run() pattern used |
| **TOTAL** | | **80 + 10★** | |

---

### Scoring Criteria

**Technical (60 pts):** Correct decorator usage, correct filter logic, correct rule-based classifier, session state implementation, requirements.txt completeness.

**Communication (20 pts):** Task 5b NRA answer — must cite specific decorator name, explain the rerun mechanism, give concrete code fix. Generic answers ("add caching") score ≤ 3/10.

---

### Key Takeaway — Day 145

**`@st.cache_data` is what separates a demo from a product.**

Without it, every widget click reruns your entire script — 500-row loads, groupby aggregations, chart renders — all from scratch. With it, the first call is slow (loading), every subsequent call is instant (memory). The pattern is:

```
utils/data_loader.py  →  @st.cache_data on all load + aggregation functions
app.py + pages/       →  import from utils (one source of truth for data)
sidebar button        →  .clear() + st.rerun() for client-facing refresh control
```

Multi-page apps built on this pattern — shared cache in utils, navigation via pages/ or st.navigation(), session state for cross-page choices — are the standard deliverable format for professional Streamlit freelance work. This is the architecture you will use for the Month 8 capstone.

---

### Interview Framing — Day 145

**Q: "How do you handle performance in Streamlit apps when clients complain it's slow?"**

> "The first thing I check is whether the data loading and aggregation functions have `@st.cache_data` applied. Streamlit reruns the entire script on every user interaction, so any uncached function — even a simple CSV read — fires repeatedly. I add `@st.cache_data` to all load and compute functions, which stores the output in memory and returns it instantly on subsequent calls. For ML models I use `@st.cache_resource` instead, which shares one model object across all sessions rather than reloading it per user. I also add a sidebar 'Refresh' button that calls `load_data.clear()` followed by `st.rerun()`, so clients can force fresh data whenever they need it. With this pattern, filter interactions that took 3–4 seconds drop to under 10 milliseconds."

---

### Submission Checklist

- [ ] `day145_app/utils/data_loader.py` — `get_raw_data()` + `get_summary_stats()` implemented
- [ ] `day145_app/app.py` — Home page with timing demo + 5 KPIs + refresh button
- [ ] `day145_app/pages/1_📊_Analysis.py` — Platform + Category charts with filter
- [ ] `day145_app/pages/2_🤖_Prediction.py` — Tier classifier + session state history
- [ ] `day145_app/requirements.txt` — All dependencies listed
- [ ] Task 5b NRA answer written in this notebook
- [ ] Screenshots: (1) Home page showing cache hit time, (2) Analysis page with filter, (3) Prediction page with history table
- [ ] ★ Bonus: st.navigation() migration screenshot
